# Taxonomic Profiling of Shotgun Metagenomes

**Tier 3 — Applied Bioinformatics | Module 26 · Notebook 1**

*Prerequisites: Module 01 (NGS Fundamentals), Module 04 (Microbial Diversity)*

---

**By the end of this notebook you will be able to:**
1. Explain the difference between 16S amplicon and shotgun metagenomics
2. Decontaminate metagenomic reads by removing host sequences
3. Classify reads taxonomically with Kraken2
4. Re-estimate species abundances with Bracken
5. Visualize community composition with Krona and bar charts


**Key resources:**
- [QIIME2 tutorials](https://docs.qiime2.org/2024.10/tutorials/)
- [Galaxy Training — Metagenomics](https://training.galaxyproject.org/training-material/topics/metagenomics/)
- [Kraken2 documentation](https://github.com/DerrickWood/kraken2)

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This workflow maps to production-style applied bioinformatics: pipeline design, model evaluation, and biological interpretation for decision-making.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Pipeline outputs are context-dependent: QC failures upstream can invalidate downstream interpretation.
- Batch effects and confounders can dominate signal if not controlled explicitly.
- Use uncertainty-aware decisions: rank candidates and keep rationale for every threshold.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

# CSV preview (DNA)
with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

# Variants preview
with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

# FASTA preview
print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

# Metadata preview
meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Module Overview](../README.md) | [Next: Functional Annotation →](./02_functional_annotation.ipynb)

---

## 1. Shotgun vs 16S Metagenomics

> Compare approaches: resolution (species/strain vs genus), cost, functional information, host contamination. Diagram of shotgun workflow.

## 2. Host Read Removal

> Align metagenomic reads to host genome (human hg38) with Bowtie2. Retain unmapped reads for downstream analysis. Report % host contamination.

In [ ]:
# Example: Remove human host reads
# !bowtie2 -x hg38 -1 sample_R1.fastq.gz -2 sample_R2.fastq.gz \
#     --un-conc-gz decontam_%.fastq.gz > /dev/null
# Reads in decontam_1.fastq.gz / decontam_2.fastq.gz are non-human

## 3. Taxonomic Classification with Kraken2

> Run Kraken2 against the standard database. Parse the output report. Show confidence score threshold effects.

In [ ]:
# Example: Kraken2 classification
# !kraken2 --db kraken2_db/ --paired --gzip-compressed \
#     decontam_1.fastq.gz decontam_2.fastq.gz \
#     --report kraken2_report.txt --output kraken2_output.txt

import pandas as pd
# Example: Parse Kraken2 report
# cols = ['pct', 'clade_reads', 'direct_reads', 'rank', 'taxid', 'name']
# report = pd.read_csv('kraken2_report.txt', sep='\t', names=cols)

## 4. Abundance Re-estimation with Bracken

> Run Bracken at species level to redistribute reads from higher taxonomy nodes. Compare Kraken2 vs Bracken species estimates.

In [ ]:
# Example: Bracken species-level re-estimation
# !bracken -d kraken2_db/ -i kraken2_report.txt -o bracken_species.txt \
#     -r 150 -l S -t 10

## 5. Community Composition Visualization

> Stacked bar chart of top 20 species across samples. Krona HTML chart. Alpha diversity (Shannon, Simpson) on Bracken abundances.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Example: Plot top 20 taxa across samples
# bracken_df = pd.read_csv('bracken_species.txt', sep='\t')
# top20 = bracken_df.nlargest(20, 'fraction_total_reads')
# top20.plot(kind='barh', x='name', y='fraction_total_reads', figsize=(10,8))
# plt.xlabel('Relative Abundance')
# plt.tight_layout()
# plt.show()

## Summary

> Recap the taxonomic profiling pipeline. Discuss database choices (standard, PlusPF, custom). Link to functional annotation and assembly notebooks.

---

[← Module Overview](../README.md) | [Next: Functional Annotation →](./02_functional_annotation.ipynb)

---